**Part 2 · LangChain** · step 6 of 23  
[← 05 Messages](05_messages.ipynb) · [🏠 Roadmap](../README.md) · [07 Structured output →](07_structured_output.ipynb)

# 06 · Prompt Templates & LCEL Chains

**What you'll learn**
- `PromptTemplate` — a reusable string with `{placeholders}`.
- `ChatPromptTemplate` — the same idea for a *list of messages*.
- `MessagesPlaceholder` — slot an entire chat history into a prompt.
- **LCEL** (LangChain Expression Language): chain steps with `|` → `prompt | model | parser`.

In [ ]:
from dotenv import load_dotenv

# Reads API keys (OPENAI_API_KEY, GROQ_API_KEY, GOOGLE_API_KEY, TAVILY_API_KEY)
# from the .env file at the AgenticAI root. load_dotenv() walks up parent
# folders until it finds one, so every notebook can share a single .env.
load_dotenv()

## 1. `PromptTemplate` — single variable

In [ ]:
from langchain_core.prompts import PromptTemplate

template = """
You are an AI Tutor.

Answer the following question.

Question :
{question}
"""

prompt = PromptTemplate.from_template(template)
result = prompt.invoke({"question": "What is kafka ??"})  # -> StringPromptValue
print(result)

text='\nYou are an AI Tutor.\n\nAnswer the following question.\n\nQuestion : \nWhat is kafka ??\n\n'


In [ ]:
# .to_string() gives the final text that would be sent to the model
print(result.to_string())


You are an AI Tutor.

Answer the following question.

Question : 
What is kafka ??




## 2. Multiple variables

In [ ]:
prompt = PromptTemplate.from_template(
    """
You are a {role}.

Explain me {topic}
in {level} level.
"""
)

prompt.invoke({"role": "Teacher", "topic": "Kafka", "level": "Hard"}).to_string()

## 3. `ChatPromptTemplate` + LCEL chain
`prompt | model | parser` creates a **chain**. Calling `chain.invoke(dict)` runs each step and pipes
the output of one into the next:

`dict` → **prompt** → messages → **model** → `AIMessage` → **StrOutputParser** → `str`

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.chat_models import init_chat_model

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "{question}"),
])
model = init_chat_model("gpt-4o-mini")
parser = StrOutputParser()  # AIMessage -> plain string

chain = prompt | model | parser

result = chain.invoke({"question": "Explain Kafka partitions."})
print(result)

Kafka partitions are a fundamental concept in Apache Kafka, which is a distributed event streaming platform. Understanding partitions is crucial for grasping how Kafka achieves scalability, fault tolerance, and high throughput. Here’s a detailed explanation of Kafka partitions:

### 1. **What is a Partition?**
A partition is a unit of storage in a Kafka topic. Each topic in Kafka is divided into one or more partitions, and these partitions are the basic building blocks of how Kafka handles data.

### 2. **Data Organization**
- **Sequential Write**: Each partition is an ordered, immutable sequence of records (messages) that is continuously appended to. A record within a partition is identified by its offset, which is a unique identifier incremented with each new message.
- **Log Structure**: Internally, each partition is implemented as a log. As messages are produced to a partition, they are written sequentially to this log, which optimizes the writing process and makes it highly effici

## 4. `MessagesPlaceholder` — inject chat history
Reserves a spot where a whole **list of messages** goes. This is how you add memory to a chain.

In [ ]:
from langchain_core.prompts import MessagesPlaceholder
from langchain.messages import HumanMessage, AIMessage

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are helpful."),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])

result = prompt.invoke({
    "history": [HumanMessage(content="Hi"), AIMessage(content="Hello")],
    "question": "Explain Kafka",
})
print(result.to_string())

## Revision notes
| Tool | Input | Output |
|---|---|---|
| `PromptTemplate` | `dict` | `StringPromptValue` (`.to_string()`) |
| `ChatPromptTemplate` | `dict` | `ChatPromptValue` (`.to_messages()`) |
| `MessagesPlaceholder("history")` | list of messages | expands in place |
| `StrOutputParser` | `AIMessage` | `str` |
| `a \| b \| c` | — | a chain: output of `a` becomes input of `b` |

---
✅ **Done with 06 Prompt templates & chains.** Next: [07 Structured output →](07_structured_output.ipynb)  
[🏠 Roadmap](../README.md)